In [1]:
import sys

print(sys.version)

# TIME SERIES TRANSFORMER WAS DONE IN PYTHON 3.10.20

3.10.20 (main, Mar 11 2026, 17:43:48) [Clang 20.1.8 ]


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D, Dropout)
from tensorflow.keras import backend as K
import gc

df = pd.read_excel('CLEANED_DATASET_THESIS_FINAL.xlsx') # CHANGE DIRECTORY TO RIGHT PATH

df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"].astype(str).str.replace("-M", "-"),format="%Y-%m").dt.to_period("M")

new_rows = []

SEQ_LEN = 12
WINDOW = 36

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i2 in range(len(X) - seq_len):
        Xs.append(X[i2:i2+seq_len])
        ys.append(y[i2+seq_len])
    return np.array(Xs), np.array(ys)

for country, group in df.groupby('COUNTRY'):

    group = group.sort_values('TIME_PERIOD').copy()

    temp = group[['COUNTRY', 'TIME_PERIOD', 'OBS_VALUE', 'GPR', 'ANNUALIZED_VOLATILITY']].copy()

    temp['OBS_VALUE'] = pd.to_numeric(temp['OBS_VALUE'], errors='coerce')
    temp['GPR'] = pd.to_numeric(temp['GPR'], errors='coerce')

    temp['LAG_3'] = temp['OBS_VALUE'].shift(3)
    temp['ROLL_MEAN_3'] = temp['OBS_VALUE'].rolling(3).mean()
    temp['ROLL_MEAN_12'] = temp['OBS_VALUE'].rolling(12).mean()
    temp['LOG_RETURN'] = np.log(temp['OBS_VALUE'] / temp['OBS_VALUE'].shift(1))

    temp['TARGET'] = temp['OBS_VALUE'].shift(-1)

    temp = temp.dropna().reset_index(drop=True)

    preds, true = [], []
    vol_preds, vol_true = [], []

    features = [
        'OBS_VALUE',
        'GPR',
        'LAG_3',
        'ROLL_MEAN_3',
        'ROLL_MEAN_12',
        'LOG_RETURN']

    start = max(SEQ_LEN + WINDOW, 24)
    STEP = 24

    for i in range(start, len(temp), STEP):

        train = temp.iloc[i-WINDOW:i].copy()
        test = temp.iloc[i:i+1].copy()

        scaler_X = MinMaxScaler()
        scaler_y = MinMaxScaler()

        X_train = train[features].values.astype(np.float32)
        y_train = train[['TARGET']].values.astype(np.float32)

        X_test = test[features].values.astype(np.float32)
        y_test = test[['TARGET']].values.astype(np.float32)

        scaler_X.fit(X_train)
        scaler_y.fit(y_train)

        X_train = scaler_X.transform(X_train)
        X_test = scaler_X.transform(X_test)

        y_train = scaler_y.transform(y_train)
        y_test = scaler_y.transform(y_test)

        X_seq, y_seq = create_sequences(X_train, y_train, SEQ_LEN)

        inputs = Input(shape=(SEQ_LEN, len(features)))

        attention = MultiHeadAttention(num_heads=2, key_dim=2)(inputs, inputs)

        x = LayerNormalization(epsilon=1e-6)(attention + inputs)

        x = Dense(64, activation='relu')(x)
        x = Dropout(0.1)(x)

        x = GlobalAveragePooling1D()(x)

        outputs = Dense(1)(x)

        model = Model(inputs, outputs)
        model.compile(optimizer='adam', loss='mse')

        model.fit(X_seq, y_seq, epochs=2, batch_size=8, verbose=0, shuffle=False)

        last_seq = X_seq[-1].reshape(1, SEQ_LEN, len(features))

        pred_scaled = model.predict(last_seq, verbose=0)

        pred = scaler_y.inverse_transform(pred_scaled)[0][0]
        actual = y_test[0][0]

        preds.append(pred)
        true.append(actual)

        obs = test['OBS_VALUE'].values[0]

        if obs > 0 and pred > 0 and actual > 0:
            vol_true.append(np.log(actual / obs))
            vol_preds.append(np.log(pred / obs))
        else:
            vol_true.append(np.nan)
            vol_preds.append(np.nan)

        K.clear_session()
        del model
        gc.collect()

    mae = mean_absolute_error(true, preds)
    mse = mean_squared_error(true, preds)

    test_eval = temp.iloc[start:start+len(preds)].copy()
    test_eval = test_eval.reset_index(drop=True)

    test_eval['PRED'] = preds
    test_eval['TRUE'] = true

    test_eval['VOL_ACT'] = vol_true
    test_eval['VOL_PRED'] = vol_preds

    vol_df = test_eval.dropna(subset=['VOL_ACT', 'VOL_PRED'])

    if len(vol_df) > 1:
        mae_vol = mean_absolute_error(vol_df['VOL_ACT'], vol_df['VOL_PRED'])
        mse_vol = mean_squared_error(vol_df['VOL_ACT'], vol_df['VOL_PRED'])
    else:
        mae_vol = np.nan
        mse_vol = np.nan

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_full = temp[features].values.astype(np.float32)
    y_full = temp[['TARGET']].values.astype(np.float32)

    scaler_X.fit(X_full)
    scaler_y.fit(y_full)

    X_full = scaler_X.transform(X_full)
    y_full = scaler_y.transform(y_full)

    X_seq, y_seq = create_sequences(X_full, y_full, SEQ_LEN)

    inputs = Input(shape=(SEQ_LEN, len(features)))

    attention = MultiHeadAttention(num_heads=2, key_dim=2)(inputs, inputs)

    x = LayerNormalization(epsilon=1e-6)(attention + inputs)

    x = Dense(64, activation='relu')(x)
    x = Dropout(0.1)(x)

    x = GlobalAveragePooling1D()(x)

    outputs = Dense(1)(x)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')

    model.fit(X_seq, y_seq, epochs=2, batch_size=8, verbose=0, shuffle=False)

    last_sequence = X_seq[-1].reshape(1, SEQ_LEN, len(features))

    pred_scaled = model.predict(last_sequence, verbose=0)
    prediction = scaler_y.inverse_transform(pred_scaled)[0][0]

    last_row = temp.iloc[-1]
    new_time = group['TIME_PERIOD'].iloc[-1] + 1

    prev_value = last_row['TARGET']

    per_change = (prediction - prev_value) / prev_value
    log_return = np.log(prediction / prev_value)

    last_11 = temp['LOG_RETURN'].dropna().iloc[-11:]
    last_12 = pd.concat([last_11, pd.Series([log_return])])

    st_dev = last_12.std()
    annualized_vol = st_dev * np.sqrt(12)

    new_rows.append({
        'COUNTRY': country,
        'TIME_PERIOD': new_time,
        'PRED_VAL_TRANS': prediction,
        'PER_CHANGE': per_change,
        'LOG_RETURN': log_return,
        'ST_DEV': st_dev,
        'ANNUALIZED_VOLATILITY': annualized_vol,
        'TRANS_PRED_MAE': mae,
        'TRANS_PRED_MSE': mse,
        'TRANS_VOL_MAE': mae_vol,
        'TRANS_VOL_MSE': mse_vol
    })

    K.clear_session()
    del model
    gc.collect()

df_new = pd.DataFrame(new_rows)
df_updated = pd.concat([df, df_new], ignore_index=True)
df_updated = df_updated.sort_values(['COUNTRY', 'TIME_PERIOD'])
df_updated.to_excel('TRANS_RW_Updated_Clean_DF.xlsx', index=False)


In [ ]:
df_metrics = pd.read_excel('TRANS_RW_Updated_Clean_DF.xlsx')

print(df_metrics['TRANS_PRED_MAE'].describe())

print(df_metrics['TRANS_PRED_MSE'].describe())

print(df_metrics['TRANS_VOL_MAE'].describe())

print(df_metrics['TRANS_VOL_MSE'].describe())